# unit06 レッスン: キャップストーン — エンドツーエンドのスクレイパー

**このレッスンで作れるようになるもの**: unit01〜05でバラバラに学んだ部品(パース / セレクタ / マナー / クレンジング / CSV)を、**1本のパイプライン**に組み上げる設計図が手に入ります。具体的には (1)「次へ」リンクを辿ってページを巡回するループ、(2) robots.txt を尊重してから取りにいくマナー層、(3) 小さな関数を合成して全体を統括する `run_pipeline` — この3つの骨格を、手を動かして組み立てます。

これがコースの総仕上げです。ここで作る形が、**そのまま実務のスクレイパーの最小形**になります。

- 所要時間: 15〜25分
- 進め方: セルを上から順に実行(`Shift+Enter`)。「書いてみる」セルだけ自分で書く
- 詰まったら: Claude に聞いてOK(答えではなくヒントをくれます)

In [ ]:
def check(name, actual, expected, hint=""):
    import numpy as _np
    try:
        ok = actual is not None and bool(_np.all(_np.isclose(_np.asarray(actual, dtype=float), _np.asarray(expected, dtype=float))))
    except (TypeError, ValueError):
        ok = actual == expected
    if ok:
        print(f"[OK] {name}: 正解!")
    else:
        print(f"[NG] {name}: 期待値 {expected!r} / 実際 {actual!r}")
        if hint:
            print(f"     ヒント: {hint}")
    return ok


from bs4 import BeautifulSoup

# --- このレッスン用の「縮小版サイト」------------------------------------------
# 演習の data/site/ は 3ページ・10商品ですが、レッスンでは仕組みが一目で追える
# 2ページ・3商品の "手のひらサイズ" のサイトを、この場でリテラルとして用意します
# (ファイルを読まないので、どこで実行しても動きます = cwd 非依存)。
# 本物の requests.get(url).text の代わりに、下の fetch(url) がこの辞書から
# HTML文字列を返す「ローカルフェッチャ」です。差し替え可能なようにこの関数越しに全取得します。

BASE_URL = "https://mini-shop.example/"
USER_AGENT = "MiniShopBot"

# 一覧ページ1(2商品 + 次ページリンクあり)
LIST_1 = """<html><body>
  <div class="grid">
    <div class="card"><a class="product-link" href="item_1.html">apple</a></div>
    <div class="card"><a class="product-link" href="item_2.html">banana</a></div>
  </div>
  <nav><a class="next-page" href="list_2.html">次へ</a></nav>
</body></html>"""

# 一覧ページ2(1商品 + 次ページリンク "なし" = ここで巡回が止まる)
LIST_2 = """<html><body>
  <div class="grid">
    <div class="card"><a class="product-link" href="item_3.html">cherry</a></div>
  </div>
  <nav><span class="current">2</span></nav>
</body></html>"""

# 詳細ページ3枚(name / price / stock を持つ)
ITEM_1 = """<html><body>
  <h1 class="item-name">apple</h1>
  <p class="item-price">120円</p>
  <p class="item-stock">在庫あり</p></body></html>"""
ITEM_2 = """<html><body>
  <h1 class="item-name">banana</h1>
  <p class="item-price">1,080円</p>
  <p class="item-stock">在庫あり</p></body></html>"""
ITEM_3 = """<html><body>
  <h1 class="item-name">cherry</h1>
  <p class="item-price">450円</p>
  <p class="item-stock">在庫切れ</p></body></html>"""

# robots.txt(MiniShopBot は /admin/ 以外OK)
ROBOTS_TXT = """User-agent: *
Disallow: /private/

User-agent: MiniShopBot
Disallow: /admin/
Crawl-delay: 1"""

# URL -> HTML文字列 の対応表。実サイトでは「サーバー上のファイル」に当たる
SITE = {
    BASE_URL + "robots.txt": ROBOTS_TXT,
    BASE_URL + "list_1.html": LIST_1,
    BASE_URL + "list_2.html": LIST_2,
    BASE_URL + "item_1.html": ITEM_1,
    BASE_URL + "item_2.html": ITEM_2,
    BASE_URL + "item_3.html": ITEM_3,
}


def fetch(url):
    """URLを受け取り、対応するHTML文字列を返すローカルフェッチャ(本物のrequests.get(url).text相当)"""
    return SITE[url]


# sleep も "注入" する。テストや学習中に本当に待たされないよう、time.sleep ではなく
# 「呼ばれた回数を記録するだけのダミー」を使う。実運用では time.sleep を渡す。
class FakeSleep:
    def __init__(self):
        self.calls = []          # 呼ばれるたびに待ち秒数を記録
    def __call__(self, seconds=0):
        self.calls.append(seconds)

print("準備OK! SITEには", len(SITE), "ページ。商品は3件、一覧は2ページ構成です。")

---
## まず全体の地図: unit01〜05の部品が、どこに座るか

総仕上げなので、新しいAPIはほとんど出てきません。仕事は **「今まで学んだ部品を1本の線につなぐ」** ことです。下の図で、各ユニットの成果物がパイプラインのどこに入るかを確認してください。

```
 start_url
    │
    ▼
[robots確認] ── robots.txt を尊重(unit04 マナー)★概念2
    │  許可されなければ何もしない
    ▼
[一覧を巡回] ── 「次へ」を辿るループ(★概念1) + セレクタでリンク収集(unit03)
    │            ページごとに sleep でレート制限(unit04)
    ▼  詳細URLのリスト
[詳細を取得] ── 1件ずつ fetch → BeautifulSoupでパース(unit02)
    │            欠損/失敗はスキップしてログ(unit05 例外処理)
    ▼  生テキストの list[dict]
[整形]      ── "1200円"→1200、"産地: X"→"X"(unit05 クレンジング)
    │
    ▼  きれいな list[dict]
[CSV出力]   ── csv.DictWriter で1本のCSVに(unit05)★概念3で全部つなぐ
```

このレッスンでは、この図の **★の3か所** を順に組み立てます。つなぎ終えると、上から下まで一気に流れる `run_pipeline` が完成します。

---
## 概念1: ページネーション巡回 —「次へ」を辿るループ

### なぜ学ぶか
実際の商品一覧は、1ページには収まりません。「1/50ページ」のように分割され、下部の「次へ」リンクを辿らないと全件は集まりません。しかも**事前に総ページ数が分からない**ことが多い(サイトが増減する)。だから「既知の全ページを `foreach` で回す」ことができず、**「次へ」が無くなるまで進む `while` ループ**が必要になります。求人票の「ページネーション対応のクローラ」はまさにこれです。

### 解説

C# で件数が分からないまま辿るなら `while` を使いますね。Python でも同じ発想です:

```
current = start_url
while current is not None:        # 次ページがある限り続ける
    html = fetch(current)          # そのページを取得
    ...リンクを集める...
    current = 次ページのURL or None # 無ければ None → ループ終了
```

ここで**新しい概念**が2つあります。

**(1) 訪問済み集合による無限ループ対策**
「次へ」が実は前のページに戻るよう壊れていると、A→B→A→B... と**無限ループ**します。対策は、訪れたURLを **`set`(集合)** に記録し、既に訪れたURLに戻ろうとしたら止めること。Python の `set` は C# の `HashSet<T>` そのもので、`visited.add(url)` で追加、`url in visited` で存在チェック(どちらも高速)。

**(2) セレクタでのリンク収集(unit03の復習)**
`soup.find_all("a", class_="product-link")` は「class が `product-link` の `<a>` タグを**全部**リストで返す」メソッド。`soup.find("a", class_="next-page")` は「最初の1個だけ、無ければ `None`」。この `None` が「次ページ無し」の合図になります。

> 補足: この後の演習(ex01/ex04)のフィクスチャは「次へ」リンクが一方向で戻りリンクを踏まない構成なので、演習では visited なしの簡略版で書けます。実務のクローラでは visited は必須です。

In [ ]:
# GOAL: 「次へ」を辿るループが、次ページが無いページで自然に止まるのを見る

# STEP 1: 1ページ分のHTMLから、商品詳細リンクを全部集める(unit03のセレクタ)
def extract_item_links(list_html):
    soup = BeautifulSoup(list_html, "html.parser")
    # class="product-link" の <a> を全部取り、href を BASE_URL と連結して絶対URLにする
    return [BASE_URL + a["href"] for a in soup.find_all("a", class_="product-link")]

# STEP 2: 次ページURLを取り出す。無ければ None(=ここで止まる合図)
def extract_next_page_url(list_html):
    soup = BeautifulSoup(list_html, "html.parser")
    link = soup.find("a", class_="next-page")
    return None if link is None else BASE_URL + link["href"]

# STEP 3: 訪問済み集合で無限ループを防ぎながら、全ページのリンクを集める while ループ
def collect_all_item_links(start_url):
    all_links = []
    visited = set()                       # C# の HashSet<string> 相当
    current = start_url
    while current is not None and current not in visited:
        visited.add(current)              # 「このURLはもう見た」と記録
        html = fetch(current)
        page_links = extract_item_links(html)
        print(f"  {current} を訪問 -> リンク {len(page_links)}件")
        all_links.extend(page_links)      # extend はリストを連結(C# の AddRange)
        current = extract_next_page_url(html)   # 次へ。無ければ None でループ終了
    return all_links

print("巡回スタート:")
links = collect_all_item_links(BASE_URL + "list_1.html")
print("集めた全リンク:", links)

### 予測してみよう

`LIST_2`(2ページ目)には `class="next-page"` のリンクが**ありません**(`current` を表示するnavしか無い)。

**実行する前に予測**: 次のセルは `extract_next_page_url(LIST_2)` を呼びます。返り値は何になり、そのため `collect_all_item_links` は何ページ目で止まったでしょうか?(上のSTEP3の出力を思い出して)

In [ ]:
# 予測してから実行!
print("LIST_1 の次ページ:", extract_next_page_url(LIST_1))
print("LIST_2 の次ページ:", extract_next_page_url(LIST_2))   # ここが止まる合図
print("集めたリンクの総数:", len(links))

`LIST_2` では `find` が `None` を返し、`while` の `current is not None` が偽になってループが自然に終わりました。「次へが無い = None = 終了」という流れが巡回の心臓です。

### 書いてみる

**課題**: 訪問済み集合 `visited` に、これまで巡回した2ページ分のURL(`list_1.html` と `list_2.html`)が入っているとします。次の `candidate` が**まだ訪問していない**URLなら `True`、**既に訪問済み**なら `False` を返す判定結果を `result1` に入れてください。

`candidate = BASE_URL + "list_1.html"` なので、これは**訪問済み** → 期待値は `False`。

ヒント(概念レベル): `set` の存在チェックは `x in visited`。「まだ訪問していない」は `not (x in visited)`。

In [ ]:
visited = {BASE_URL + "list_1.html", BASE_URL + "list_2.html"}
candidate = BASE_URL + "list_1.html"

result1 = None
# ここに書く(result1 に代入する。candidate がまだ未訪問かどうかの True/False)


check("概念1: 訪問済み判定", result1, False,
      hint='candidate は visited に入っている(訪問済み)。"未訪問か?" は not (candidate in visited)')

---
## 概念2: robots対応フェッチ — マナー層を「関数」として分離する

### なぜ学ぶか
スクレイピングは「取れれば何でも取っていい」わけではありません。多くのサイトは `robots.txt` で「ここは巡回しないで」という意思表示をしています(unit04で学んだ通り)。実務では**取得の前に必ず許可を確認**するのが最低限のマナーであり、規約違反はアカウント停止や法的リスクに直結します。ここでは、その確認処理を **1つの関数に閉じ込める** 設計を学びます。

### 解説

マナーの判定を、パイプラインの本流にベタ書きすると、後で読めなくなります。そこで **「許可されているか?」だけを答える小さな関数** に切り出します。これは C# で言えば `IRobotsChecker.IsAllowed(url)` のような**責務を1つに絞ったサービス**を用意するのと同じ発想です。

判定には unit04 で使った `RobotFileParser` を使います:

- `urllib.robotparser.RobotFileParser()` … robots.txt を解釈するオブジェクトを作る
- `rp.parse(テキストの行リスト)` … robots.txt の中身(文字列)を**行のリスト**にして読み込ませる。`"...".splitlines()` で行リストになる(通常はURLから読むAPIだが、手元の文字列を食わせるにはこの `parse` を使う)
- `rp.can_fetch(ユーザーエージェント名, url)` … その User-Agent がその URL を取得してよいなら `True`

**設計のキモ**: このあと巡回ループを書くとき、本流のコードは `if is_allowed(...)` と**一言呼ぶだけ**で済みます。robots の細かい仕様(`parse` に行リストを渡す等)は関数の中に隠れ、本流は「マナーを確認してから進む」という**意図だけ**が読めるようになります。

In [ ]:
# GOAL: robotsの判定を1つの関数に閉じ込め、本流からは一言で呼べる形にする

import urllib.robotparser

# マナー層: 「USER_AGENT はこのURLを取得してよいか?」だけを答える関数
def is_allowed(robots_txt, target_url):
    rp = urllib.robotparser.RobotFileParser()
    rp.parse(robots_txt.splitlines())        # 文字列を行リストにして読み込ませる
    return rp.can_fetch(USER_AGENT, target_url)

# ROBOTS_TXT では MiniShopBot に対し Disallow: /admin/ だけ。それ以外はOK
allowed_url = BASE_URL + "list_1.html"       # 一覧ページ -> 許可されるはず
blocked_url = BASE_URL + "admin/secret.html" # 管理画面 -> 拒否されるはず

print("一覧ページは取得OK? :", is_allowed(ROBOTS_TXT, allowed_url))
print("管理画面は取得OK?   :", is_allowed(ROBOTS_TXT, blocked_url))

# 本流はこう「一言」で書ける(意図だけが読める)
if is_allowed(ROBOTS_TXT, allowed_url):
    print("-> 許可されたので、これから巡回してよい")

### 予測してみよう

`ROBOTS_TXT` の `MiniShopBot` セクションは `Disallow: /admin/` だけです。`/private/` を禁止しているのは `User-agent: *`(その他全員)のセクションで、`MiniShopBot` には**個別セクションがあるので `*` のルールは適用されません**(robots.txt は「最も具体的に一致した1グループ」だけを見る)。

**実行する前に予測**: `MiniShopBot` にとって、`/private/data.html` と `/admin/x.html` はそれぞれ取得OK(`True`)でしょうか?

In [ ]:
# 予測してから実行!
print("/private/ は取得OK?:", is_allowed(ROBOTS_TXT, BASE_URL + "private/data.html"))
print("/admin/ は取得OK?  :", is_allowed(ROBOTS_TXT, BASE_URL + "admin/x.html"))

`MiniShopBot` 専用セクションがあるため、`*` の `Disallow: /private/` は無視され `/private/` は取得OK。一方 `/admin/` は自分のセクションで禁止されているので不可。この判定ロジックが `is_allowed` の中に閉じているので、本流は結果の `True/False` だけ気にすれば済みます。

### 書いてみる

**課題**: `is_allowed` を使って、詳細ページ `item_1.html` を `MiniShopBot` が取得してよいか判定した結果(`True`/`False`)を `result2` に入れてください。`item_1.html` は禁止されていないので期待値は `True`。

ヒント(概念レベル): `is_allowed(ROBOTS_TXT, <対象URL>)` を呼ぶだけ。対象URLは `BASE_URL + "item_1.html"`。

In [ ]:
result2 = None
# ここに書く(result2 に代入する。item_1.html が取得OKかの True/False)


check("概念2: robots判定", result2, True,
      hint='is_allowed(ROBOTS_TXT, BASE_URL + "item_1.html") を呼ぶ。item_1 は禁止対象外なので True')

---
## 概念3: パイプライン組み立て — 小さな関数の「合成」

### なぜ学ぶか
ここまでで「巡回」「マナー確認」の部品ができました。残りは、詳細取得・整形・CSV出力を含めて **全部を1本につなぐ** ことです。ここで大事なのは「どう書くか」より **「どう分けるか」**。実務でスクレイパーを引き継いだり直したりするのは日常で、**保守しやすい分け方**を知っているかどうかが分かれ目になります。

### 解説

**「大きな1関数」がなぜ悪いか。** 取得もパースも整形もCSVも1つの関数に詰め込むと、次の問題が起きます:

- **テストできない**: 一部だけ(例: 整形ロジックだけ)を確かめられない。全部動かすしかない
- **直すと壊れる**: CSVの列を増やそうとしたら、無関係な巡回ループまで読む羽目になる
- **再利用できない**: 「整形だけ他でも使いたい」ができない

だから **小さな関数に分け、それを順番に呼ぶだけの「統括関数(オーケストレーター)」** を置きます。C# で言えば、各処理を担うサービスクラスを、1つの**アプリケーションサービス**が順番に呼ぶ構造そのもの:

```
run_pipeline (統括 = オーケストレーター)
   ├─ is_allowed(...)              # 概念2(マナー)
   ├─ collect_all_item_links(...)  # 概念1(巡回)
   ├─ fetch_and_clean_all(...)     # 詳細取得+整形(unit02/05)
   └─ write_catalog_csv(...)       # CSV出力(unit05)
```

統括関数は **「誰を・どの順で呼ぶか」だけ** を書きます。各部品の中身は知らなくてよい。これが読みやすさと保守性の源です。

不足している部品(詳細取得+整形、CSV出力)は次のworked exampleで用意し、最後に全部を `run_pipeline` で合成します。

In [ ]:
# GOAL: 詳細取得+整形+CSV出力の部品を用意し、統括関数で全部を"合成"する
import csv, io

# --- 部品A: 詳細ページ1枚 -> 生テキストの辞書(unit02のパース) ---
def parse_item_detail(detail_html):
    soup = BeautifulSoup(detail_html, "html.parser")
    return {
        "name":  soup.find(class_="item-name").get_text(strip=True),
        "price": soup.find(class_="item-price").get_text(strip=True),
        "stock": soup.find(class_="item-stock").get_text(strip=True),
    }

# --- 部品B: "1,080円" -> 1080 の整形(unit05のクレンジング) ---
def price_to_int(price_text):
    return int(price_text.replace("円", "").replace(",", "").strip())

# --- 部品C: 生辞書 + id -> きれいな辞書 ---
def clean_item(item_id, raw):
    return {"id": item_id, "name": raw["name"],
            "price": price_to_int(raw["price"]), "stock": raw["stock"]}

# --- 部品D: URL群を1件ずつ取得+整形。失敗はスキップしてログ、sleepは注入 ---
def fetch_and_clean_all(item_urls, sleep_fn, on_error=None):
    rows = []
    for item_id, url in enumerate(item_urls, start=1):   # id は1始まり
        try:
            raw = parse_item_detail(fetch(url))
            rows.append(clean_item(item_id, raw))
        except Exception as exc:                          # 1件の失敗で全体を止めない
            if on_error is not None:
                on_error(url, exc)
        finally:
            sleep_fn(1)                                   # 礼儀正しく1秒待つ(注入)
    return rows

# --- 部品E: きれいな辞書のリスト -> CSV文字列(unit05のDictWriter) ---
def write_catalog_csv(rows):
    buf = io.StringIO()
    writer = csv.DictWriter(buf, fieldnames=["id", "name", "price", "stock"])
    writer.writeheader()
    writer.writerows(rows)
    return buf.getvalue()

# --- 統括(オーケストレーター): 誰を・どの順で呼ぶかだけを書く ---
def run_pipeline(start_url, sleep_fn):
    # STEP 1: マナー確認(概念2)。ダメなら空で終了
    robots_txt = fetch(BASE_URL + "robots.txt")
    if not is_allowed(robots_txt, start_url):
        return []
    # STEP 2: 一覧を巡回してリンク収集(概念1)
    item_urls = collect_all_item_links(start_url)
    # STEP 3: 詳細を取得して整形(部品D)
    return fetch_and_clean_all(item_urls, sleep_fn)

sleep = FakeSleep()
catalog = run_pipeline(BASE_URL + "list_1.html", sleep)
print("整形済みカタログ:")
for row in catalog:
    print("  ", row)
print("sleepが呼ばれた回数:", len(sleep.calls), "(=詳細3件ぶん礼儀正しく待った)")
print("CSV出力:")
print(write_catalog_csv(catalog))

### 予測してみよう

`run_pipeline` の最初で robots を確認しています。もし `start_url` が**取得を禁止されたURL**だったら、`if not is_allowed(...)` が真になり `return []` で即終了します。

**実行する前に予測**: 次のセルは禁止URL `admin/list.html` を `start_url` に渡して `run_pipeline` を呼びます。返ってくるカタログの中身は何件でしょう? また、そのとき `collect_all_item_links`(巡回)は**呼ばれる**でしょうか?

In [ ]:
# 予測してから実行!
sleep2 = FakeSleep()
blocked_catalog = run_pipeline(BASE_URL + "admin/list.html", sleep2)
print("禁止URLでのカタログ件数:", len(blocked_catalog))
print("sleepが呼ばれた回数    :", len(sleep2.calls), "(0なら巡回に進んでいない)")

robotsで弾かれると `return []` で即座に終わり、巡回も詳細取得も**一切行われません**(sleep 0回)。マナー確認を**先頭に置いた**からこそ、禁止されたサイトへ無駄なアクセスをしないで済む — これが「順番」の設計の意味です。

### 書いてみる

**課題**: 上の部品を組み合わせて、**詳細URL3件から整形済みカタログを作り、その合計金額**を求めて `result3` に入れてください。手順は (1) `item_urls` を `fetch_and_clean_all` に渡してカタログを得る → (2) 各行の `"price"` を合計。3商品の価格は 120 + 1080 + 450 なので期待値は `1650`。

ヒント(概念レベル): カタログは `fetch_and_clean_all(item_urls, FakeSleep())`。合計は `sum(row["price"] for row in カタログ)`。

In [ ]:
item_urls = [BASE_URL + "item_1.html", BASE_URL + "item_2.html", BASE_URL + "item_3.html"]

result3 = None
# ここに書く(result3 に代入する。カタログを作って price を合計する)


check("概念3: パイプライン合成", result3, 1650,
      hint='rows = fetch_and_clean_all(item_urls, FakeSleep()); result3 = sum(r["price"] for r in rows)')

---
## 振り返り(1〜2文でOK — このセルを編集して書き込んでください)

- **今日学んだことを自分の言葉で**:
- **難しかったこと(あれば)**:

(この記述はセッション終了時にチューターが学習ノートとスキルレベル判定に使います)

## まとめと次へ

| 概念 | 一言で | C#で言うと |
|------|--------|-----------|
| ページネーション巡回 | 「次へ」が None になるまで `while` で辿る。`set` で無限ループ防止 | `HashSet<T>` で訪問管理する `while` |
| robots対応フェッチ | マナー判定を `is_allowed` 1関数に閉じ込め、本流は一言で呼ぶ | 責務を絞った `IRobotsChecker` サービス |
| パイプライン組み立て | 小さな関数を「統括関数」が順番に呼ぶ。大きな1関数は保守で死ぬ | 各サービスを順に呼ぶアプリケーションサービス |

**この形が、そのまま実務のスクレイパーの最小形です。** 今日のレッスンは2ページ・3商品の"手のひらサイズ"でしたが、演習の `data/site/`(3ページ・10商品)に差し替えても、同じ骨格で動きます — 変わるのはページ数と商品数だけ。それが「小さく分けて統括する」設計の強さです。

**この先どこで使うか**:
- **演習 ex01〜ex04** で、この骨格を本物の `data/site/` フィクスチャに対して自分の手で完成させます。ex01=巡回、ex02=詳細取得、ex03=整形+CSV、ex04=全部を `run_pipeline` に統合(まさに概念3)。
- **実務の次の一歩**: ここに「取得失敗時のリトライ(指数バックオフ)」「並行取得」「取得済みの差分更新」を足せば、そのまま運用スクレイパーになります。今日組んだ `run_pipeline` は、その土台です。

**次**: 演習 `ex01_collect_links.py` へ。lesson を見ながらで OK。テストは
`python -m pytest courses/web-scraping/unit06-capstone-scraper/tests/test_ex01.py -q`